In [1]:
!pip install -q transformers datasets seqeval accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import numpy as np
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, set_seed
from seqeval.metrics import precision_score, recall_score, f1_score

set_seed(42)

raw = load_dataset("tomaarsen/conll2003")
label_list = raw["train"].features["ner_tags"].feature.names
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

model_name = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align(examples):
    tokenized = tokenizer(examples["tokens"], truncation=True,
                          is_split_into_words=True)
    all_labels = []
    for i, tags in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev = None
        label_ids = []
        for w in word_ids:
            if w is None:
                label_ids.append(-100)
            elif w != prev:
                label_ids.append(tags[w])
            else:
                label_ids.append(-100)
            prev = w
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized

tokenized_ds = raw.map(tokenize_and_align, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)
    true_preds, true_labels = [], []
    for pred_row, label_row in zip(preds, labels):
        p_tags, l_tags = [], []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                p_tags.append(label_list[p])
                l_tags.append(label_list[l])
        true_preds.append(p_tags)
        true_labels.append(l_tags)
    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }

README.md:   0%|          | 0.00/15.0k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.24MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  316kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (8 outputs left out)

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [3]:
# Baseline model from 01_baseline.ipynb, hosted on Hugging Face
BASELINE_ID = "Yxian909/ner-baseline"

In [4]:
train_lines = [
    "[Siti Nurhaliza](PER) performed at [Dewan Filharmonik Petronas](ORG) in [Kuala Lumpur](LOC).",
    "Dr. [Ravi Kumar](PER) works at [Hospital Sultanah Aminah](ORG) in [Johor Bahru](LOC).",
    "[Nur Aisyah binti Ahmad](PER) studies at [UTM](ORG).",
    "[Ahmad bin Ali](PER) bekerja di [Maybank](ORG) di [Kuala Lumpur](LOC).",
    "[Tan Mei Ling](PER) moved from [Penang](LOC) to [Kuching](LOC) last year.",
    "Encik [Razak bin Hassan](PER) is the manager at [Tenaga Nasional](ORG).",
    "[Lee Chong Wei](PER) trained at [Bukit Jalil](LOC) for many years.",
    "[Michelle Yeoh](PER) was born in [Ipoh](LOC).",
    "Puan [Fatimah binti Yusof](PER) teaches at [SMK Tun Fatimah](ORG) in [Johor Bahru](LOC).",
    "[AirAsia](ORG) flights from [Kuching](LOC) to [Kota Kinabalu](LOC) are fully booked.",
    "[Muthu a/l Selvam](PER) joined [Petronas](ORG) in [Miri](LOC).",
    "The conference will be held in [Putrajaya](LOC) next month.",
    "[Wong Kah Hou](PER) studies at [Universiti Malaya](ORG).",
    "Saya tinggal di [Skudai](LOC) berhampiran [Universiti Teknologi Malaysia](ORG).",
    "[Nicol David](PER) won the squash title in [Penang](LOC).",
    "Cik [Aina binti Rosli](PER) works at [Bank Negara Malaysia](ORG).",
    "[Grab](ORG) drivers in [Kuala Lumpur](LOC) asked for higher fares.",
    "[Chong Yi Xuan](PER) visited [Sibu](LOC) with her family.",
    "The patient was sent to [Hospital Umum Sarawak](ORG) in [Kuching](LOC).",
    "[Kementerian Kesihatan Malaysia](ORG) mengumumkan kempen baharu.",
    "[Arun Pillai](PER) flew from [Kuala Lumpur](LOC) to [Melaka](LOC).",
    "[Yuna](PER) will perform in [George Town](LOC) on Saturday.",
    "[Farhan bin Ismail](PER) is an engineer at [Proton](ORG) in [Shah Alam](LOC).",
    "Dr. [Lim Soo Hock](PER) joined [Sunway Medical Centre](ORG).",
    "[Nurul Izzah binti Kamal](PER) lives in [Seremban](LOC).",
    "The students from [Politeknik Kuching](ORG) visited [Bako National Park](LOC).",
    "[Kavitha Raj](PER) started a new job at [Shopee](ORG).",
    "[Ali bin Abu](PER) membeli kereta di [Ipoh](LOC).",
    "[Ahmad Zaki bin Roslan](PER) dari [Alor Setar](LOC) bekerja di [Telekom Malaysia](ORG).",
    "We had lunch together yesterday.",
    "The weather in the afternoon was very hot.",
    "Hari ini hujan lebat.",
    "Please submit your report before Friday.",
    "[Lau Hui Min](PER) is the president of the robotics club at [UTM](ORG).",
    "[Sarawak](LOC) is known for its rainforests and wildlife.",
    "[Ng Wei Hong](PER) opened a cafe in [Miri](LOC).",
    "[Hospital Pulau Pinang](ORG) is located in [George Town](LOC).",
    "[Hafiz bin Mahmud](PER) dan [Syafiqah binti Daud](PER) bertemu di [Kuantan](LOC).",
    "The meeting with [CIMB](ORG) is scheduled for Tuesday.",
    "[Priya Nair](PER) graduated from [Universiti Sains Malaysia](ORG).",
    "Encik [Jamal bin Osman](PER) from [Kota Bharu](LOC) called the helpdesk.",
    "[Rina Marlina](PER) works for [Petronas](ORG) in [Bintulu](LOC).",
    "[Khairul Amri](PER) joined [Sime Darby](ORG) in [Kuala Lumpur](LOC).",
    "[Lim Jia Yi](PER) is from [Sarikei](LOC).",
    "[Maxis](ORG) and [Celcom](ORG) offer new data plans.",
    "[Sofea binti Hamid](PER) works at [Hospital Kuala Lumpur](ORG).",
    "The festival in [Kuching](LOC) attracts many tourists.",
    "[Daniel Wong](PER) is studying in [Melbourne](LOC) now.",
    "[Angie Shanon](PER) is studying at [Melbourne University](ORG) now.",
    "[Zainal bin Ariffin](PER) is the headmaster of [SMK Batu Lintang](ORG).",
    "[Gopal Krishnan](PER) lives near [Taman Universiti](LOC) in [Skudai](LOC).",
    "[Aisyah](PER) dan [Hakim](PER) pergi ke [Langkawi](LOC).",
    "[Tan Chee Keong](PER) represents [Universiti Malaysia Sarawak](ORG) in the debate.",
    "[Intan Syafiqah](PER) works at [Khazanah Nasional](ORG).",
    "Thank you for your message.",
    "[Lucas anak Jugah](PER) comes from [Sri Aman](LOC) in [Sarawak](LOC).",
    "[Jessica Ling](PER) visited [Penang](LOC) with colleagues from [Intel](ORG).",
    "[Mohd Rizal bin Salleh](PER) teaches at [Universiti Teknologi MARA](ORG).",
    "[Siew Ling](PER) bought tickets for [Malaysia Airlines](ORG).",
    "[Pooja Devi](PER) will move to [Cyberjaya](LOC) next year.",
    "Kedai itu dibuka pada pukul sembilan pagi.",
    "[Ceylonese](ORG) is a famous restaurant in [Kuching](LOC).",
    "[Amirul Hakim](PER) works at [Intel](ORG) in [Penang](LOC).",
    "[Chua Wei Jian](PER) graduated from [Universiti Kebangsaan Malaysia](ORG).",
    "Dr. [Nadia Farhana](PER) conducts research at [Universiti Malaya](ORG).",
    "[Kumaravel Rajan](PER) moved to [Kuala Lumpur](LOC) for work.",
    "[Nur Amirah](PER) is an employee at [Maybank](ORG) in [Johor Bahru](LOC).",
    "[Fong Jia Wen](PER) visited [Kota Kinabalu](LOC) last month.",
    "[Muhammad Firdaus](PER) studies engineering at [Universiti Teknologi PETRONAS](ORG).",
    "[Saraswathy Devi](PER) works at [Hospital Queen Elizabeth](ORG) in [Kota Kinabalu](LOC).",
    "[Daniel anak Mawan](PER) comes from [Kapit](LOC) in [Sarawak](LOC).",
    "[Siti Hajar](PER) attended a seminar in [Putrajaya](LOC).",
    "[Jason Lim](PER) joined [Deloitte](ORG) in [Kuala Lumpur](LOC).",
    "[Aqilah binti Zainuddin](PER) teaches at [Universiti Pendidikan Sultan Idris](ORG).",
    "[Ramesh Kumar](PER) bought a house in [Shah Alam](LOC).",
    "[Chong Wen Qi](PER) works for [Samsung](ORG) in [Penang](LOC).",
    "[Nur Syafiqah](PER) travelled from [Miri](LOC) to [Kuching](LOC).",
    "[Haziq bin Azman](PER) is studying at [Universiti Malaysia Pahang Al-Sultan Abdullah](ORG).",
    "[Mei Xin](PER) started an internship at [Petronas](ORG).",
    "[Arvind Singh](PER) lives in [Seremban](LOC).",
    "[Farah Nadia](PER) attended a conference organised by [MDEC](ORG).",
    "[Lim Chee Seng](PER) works at [CIMB](ORG) in [Melaka](LOC).",
    "[Aiman Zulkifli](PER) visited [Gunung Mulu National Park](LOC).",
    "[Priyanka Menon](PER) graduated from [Universiti Sains Malaysia](ORG).",
    "[Syed Danish](PER) joined [AirAsia](ORG) as an engineer.",
    "[Lee Xin Yi](PER) moved from [Ipoh](LOC) to [Cyberjaya](LOC).",
    "[Mohd Azlan bin Ismail](PER) works at [Tenaga Nasional](ORG).",
    "[Tan Siew Mei](PER) attended a workshop in [Kuching](LOC).",
    "[Kavinesh Raj](PER) is a researcher at [Malaysian Palm Oil Board](ORG).",
    "[Nur Iman](PER) studies computer science at [Universiti Malaysia Sarawak](ORG).",
    "[Wong Zi Hao](PER) visited [Batu Caves](LOC) during the holidays.",
    "[Hakimah binti Salleh](PER) works at [Bank Negara Malaysia](ORG).",
    "[Faizal bin Rahman](PER) travelled to [Bintulu](LOC) for a business meeting.",
    "[Rachel Tan](PER) joined [Shopee](ORG) in [Kuala Lumpur](LOC).",
    "[Muhammad Danish](PER) attended a lecture at [Universiti Putra Malaysia](ORG).",
    "[Siti Khadijah](PER) is from [Kota Bharu](LOC).",
    "[Arun Kumar](PER) works at [Proton](ORG) in [Shah Alam](LOC).",
    "[Yap Mei Ling](PER) visited [Langkawi](LOC) with her friends.",
    "[Nurul Ain](PER) joined [Maxis](ORG) as a software developer.",
    "[Goh Jun Kai](PER) studies at [Universiti Tunku Abdul Rahman](ORG).",
    "[Amir Syahmi](PER) travelled from [Kuantan](LOC) to [Kuala Terengganu](LOC).",
    "[Deepa Krishnan](PER) works at [Sunway Medical Centre](ORG).",
    "[Faris bin Abdullah](PER) attended a training programme in [Putrajaya](LOC).",
    "[Ng Jia Hui](PER) works for [Intel](ORG) in [Penang](LOC).",
    "[Mohamad Irfan](PER) enrolled at [Universiti Teknologi MARA](ORG).",
    "[Shalini Devi](PER) lives near [Johor Bahru](LOC).",
    "[Lee Zhi Yang](PER) works at [Grab](ORG) in [Kuala Lumpur](LOC).",
    "[Nur Maisarah](PER) visited [Taman Negara](LOC) last summer.",
    "[Kenny Wong](PER) joined [CelcomDigi](ORG) in [Petaling Jaya](LOC).",
    "[Siti Nurul](PER) attended a meeting at [Kementerian Pendidikan Malaysia](ORG).",
    "[Viknesh Raj](PER) moved to [George Town](LOC) last year.",
    "[Alicia Chong](PER) graduated from [Universiti Malaysia Sabah](ORG).",
    "[Nur Afiqah binti Rahman](PER), who graduated from [Universiti Malaya](ORG), recently joined [Petronas](ORG) as a data analyst in [Kuala Lumpur](LOC).",
    "After spending three years at [Intel](ORG) in [Penang](LOC), [Tan Wei Jian](PER) accepted a senior engineering position at [Samsung](ORG) in [Johor Bahru](LOC).",
    "Dr. [Kavitha Raj](PER), a medical researcher at [Universiti Sains Malaysia](ORG), presented her findings at a conference held in [Kuching](LOC).",
    "Although [Muhammad Danish](PER) was born in [Kota Bharu](LOC), he completed his degree at [Universiti Teknologi Malaysia](ORG) before starting work at [Maybank](ORG).",
    "[Lee Xin Hui](PER) and [Arun Kumar](PER), who both studied at [Universiti Putra Malaysia](ORG), travelled to [Cyberjaya](LOC) to attend a technology exhibition organized by [MDEC](ORG).",
    "The delegation from [Universiti Malaysia Sarawak](ORG), led by Professor [Wong Chee Ming](PER), visited several research laboratories in [Penang](LOC) before returning to [Kuching](LOC).",
    "When [Siti Nur Amirah](PER) relocated from [Miri](LOC) to [Kuala Lumpur](LOC), she transferred from [Curtin University Malaysia](ORG) to [Universiti Malaya](ORG).",
    "[Ahmad Firdaus bin Ismail](PER), who previously worked at [Tenaga Nasional](ORG) in [Ipoh](LOC), was appointed regional manager for the company's operations in [Kuantan](LOC).",
    "Representatives from [AirAsia](ORG), [Malaysia Airlines](ORG), and [Batik Air](ORG) attended an aviation conference in [Kuala Lumpur](LOC), where [Daniel Wong](PER) presented a proposal.",
    "After completing her internship at [Deloitte](ORG) in [Kuala Lumpur](LOC), [Priya Nair](PER) received an offer to join [EY](ORG) as a financial consultant.",
    "The research team, which included Dr. [Lim Soo Hock](PER) from [Universiti Kebangsaan Malaysia](ORG), collected environmental data from several rivers in [Sarawak](LOC).",
    "[Chong Yi Xuan](PER), whose family lives in [Sibu](LOC), travelled to [Shah Alam](LOC) after receiving an admission offer from [Universiti Teknologi MARA](ORG).",
    "During the meeting in [Putrajaya](LOC), [Kementerian Kesihatan Malaysia](ORG) representatives discussed a collaboration with [Hospital Umum Sarawak](ORG) and researchers from [Universiti Malaysia Sabah](ORG).",
    "The software development team at [Grab](ORG), which operates across several Malaysian cities, appointed [Mohd Rizal bin Salleh](PER) to oversee its new office in [Johor Bahru](LOC).",
    "Before joining [Proton](ORG) in [Shah Alam](LOC), [Ng Wei Hong](PER) completed an automotive engineering programme at [Universiti Malaysia Pahang Al-Sultan Abdullah](ORG).",
    "[Nurul Izzah binti Kamal](PER), who currently lives in [Seremban](LOC), was selected to represent [Universiti Kebangsaan Malaysia](ORG) at an international competition in [Singapore](LOC).",
    "While attending a workshop organized by [Bank Negara Malaysia](ORG) in [Kuala Lumpur](LOC), [Jason Lim](PER) met financial experts from several international institutions.",
    "The hospital director, Dr. [Suresh Menon](PER), announced that [Hospital Sultanah Aminah](ORG) in [Johor Bahru](LOC) would introduce a new digital appointment system next year.",
    "Following the announcement by [Telekom Malaysia](ORG), [Aina binti Rosli](PER) travelled from [Kota Kinabalu](LOC) to [Kuching](LOC) to attend a regional telecommunications meeting.",
    "[Gopal Krishnan](PER), who teaches computer science at [Universiti Teknologi Malaysia](ORG), collaborated with researchers from [Universiti Sains Malaysia](ORG) on a project based in [Penang](LOC).",
    "Selepas menamatkan pengajian di [Universiti Putra Malaysia](ORG), [Nur Syafiqah binti Ahmad](PER) menerima tawaran pekerjaan daripada [Petronas](ORG) di [Bintulu](LOC).",
    "[Ahmad Zaki bin Roslan](PER), yang berasal dari [Alor Setar](LOC), berpindah ke [Cyberjaya](LOC) selepas dilantik sebagai jurutera di [Telekom Malaysia](ORG).",
    "Walaupun [Kavitha Raj](PER) dibesarkan di [Ipoh](LOC), beliau memilih untuk melanjutkan pelajaran di [Universiti Malaysia Sabah](ORG) sebelum bekerja di [Kota Kinabalu](LOC).",
    "Dalam persidangan yang berlangsung di [Putrajaya](LOC), wakil daripada [Kementerian Pendidikan Malaysia](ORG) bertemu dengan Dr. [Nadia Farhana](PER) untuk membincangkan kerjasama penyelidikan.",
    "[Lim Jia Yi](PER), yang bekerja di [Maybank](ORG) di [Kuala Lumpur](LOC), telah dipilih untuk menyertai program latihan kepimpinan yang dianjurkan oleh syarikat tersebut.",
    "Selepas berpindah dari [Kuching](LOC) ke [Johor Bahru](LOC), [Farhan bin Ismail](PER) menyambung pengajian dalam bidang kejuruteraan di [Universiti Teknologi Malaysia](ORG).",
    "Pasukan penyelidik daripada [Universiti Malaysia Sarawak](ORG), yang diketuai oleh Profesor [Tan Chee Keong](PER), menjalankan kajian biodiversiti di [Taman Negara Bako](LOC).",
    "Ketika menghadiri seminar di [Melaka](LOC), [Siti Hajar](PER) bertemu dengan pegawai daripada [Bank Negara Malaysia](ORG) yang sedang mencari peserta untuk program baharu mereka.",
    "[Muhammad Afiq](PER) dilantik sebagai pengurus projek di [Sime Darby](ORG) selepas menamatkan latihan profesionalnya di [Kuala Lumpur](LOC).",
    "Pihak [Hospital Pulau Pinang](ORG) bekerjasama dengan [Universiti Sains Malaysia](ORG) untuk menjalankan kajian kesihatan yang melibatkan penduduk di sekitar [George Town](LOC).",
    "[Rachel Tan](PER), [Jason Teo](PER), and [Priya Nair](PER) represented [Intel](ORG) at a semiconductor technology exhibition in [Penang](LOC).",
    "A joint research project involving [Petronas](ORG), [Universiti Teknologi PETRONAS](ORG), and [Universiti Malaya](ORG) was launched in [Perak](LOC) under the supervision of Dr. [Ravi Kumar](PER).",
    "The new manufacturing facility established by [Proton](ORG) in [Tanjung Malim](LOC) will recruit graduates from [Universiti Malaysia Pahang Al-Sultan Abdullah](ORG), according to [Farid Hakim](PER).",
    "After meeting [Michelle Yeoh](PER) at an event in [Kuala Lumpur](LOC), the students from [Universiti Kebangsaan Malaysia](ORG) discussed opportunities for young Malaysian filmmakers.",
    "The annual technology forum in [Kuching](LOC) brought together engineers from [Huawei](ORG), [Samsung](ORG), and [Intel](ORG), including [Wong Zi Hao](PER), who represented his research team.",
    "It was Dr. [Suresh Menon](PER) who informed [Hospital Selayang](ORG) that the research team would arrive in [Selangor](LOC) on Monday.",
    "According to [Nur Amirah](PER), the new branch of [CIMB](ORG) in [Kota Kinabalu](LOC) will provide additional services to customers from surrounding districts.",
    "Although the conference was initially scheduled for [Kuala Lumpur](LOC), [MDEC](ORG) later confirmed that [Daniel Wong](PER) would deliver his presentation in [Cyberjaya](LOC).",
    "Not only did [Arun Pillai](PER) represent [Petronas](ORG) at the conference in [Bintulu](LOC), but he also introduced a new energy monitoring system developed by his team.",
    "Datuk Seri [Razlan bin Hamid](PER) told the [Education Ministry](ORG) to take firm action against school bullying.",
    "[KUALA LUMPUR](LOC): The promotion of community nurses will be discussed when the [Dewan Negara](ORG) meets on Nov 30.",
    "Senator Dr [Kumaran Pillai](PER) said nurses' welfare and career growth must be strengthened.",
    "Nurses with an SPM certificate and about two and a half years of training should be considered for a higher grade.",
    "[Aisyah](PER) works at [Petronas](ORG) in [Miri](LOC).",
    "[Kedai Ah Seng](ORG) di [Kuching](LOC) dibuka pukul 8 pagi.",
    "[Amir](PER) mengambil bas ke [Johor Bahru](LOC) setiap hari.",
    "[Siti](PER) membeli buku di [George Town](LOC), [Pulau Pinang](LOC).",
    "[Hakim](PER) menjual kuih di [Pasar Siti Khadijah](LOC), [Kota Bharu](LOC).",
    "Encik [Razak](PER) mengurus sebuah kafe di [Kota Kinabalu](LOC).",
    "[Nurul](PER) menghadiri kursus di [Universiti Malaya](ORG).",
    "[Lee Wei](PER) bermain badminton di [Ipoh](LOC).",
    "Keluarga [Farid](PER) bercuti di [Langkawi](LOC) pada hujung minggu.",
    "[Mei Ling](PER) bekerja di [AirAsia](ORG) di [Subang](LOC).",
    "Tun Dr. [Hassan bin Idris](PER) pernah menjadi Perdana Menteri [Malaysia](LOC).",
    "[Nicol David](PER) memberi inspirasi kepada ramai atlet muda.",
    "[Ravi](PER) menaiki feri dari [Butterworth](LOC) ke [Pulau Pinang](LOC).",
    "Puan [Sofia](PER) membuka butik kecil di [Melaka](LOC).",
    "[Azlan](PER) menghadiri mesyuarat di [Cyberjaya](LOC).",
    "[Dewi](PER) membeli ikan segar di [Sandakan](LOC).",
    "[Ahmad](PER) dan [Farah](PER) bertemu di [Dataran Merdeka](LOC).",
    "[Syarikat Maju Jaya Sdn. Bhd.](ORG) mempunyai pejabat di [Shah Alam](LOC).",
    "[Kumar](PER) menikmati teh tarik di [Taiping](LOC).",
    "[Zara](PER) melawat [Muzium Sarawak](LOC) bersama rakan-rakannya.",
    "[Jonathan anak Ringgit](PER) berasal dari [Kapit](LOC) tetapi bekerja di [Bintulu](LOC).",
    "[Rosnah binti Ahmad](PER) is a nurse at [Hospital Miri](ORG).",
    "[KUCHING](LOC): Heavy rain caused flash floods in several areas yesterday.",
    "[JOHOR BAHRU](LOC): The state government will open a new library next year.",
    "[Andrew Lau](PER) and his team from [Hong Leong Bank](ORG) visited [Sibu](LOC).",
    "Weh, [Jason](PER) cakap dia nak pergi [Tawau](LOC) minggu depan.",
    "Do you know where [Hospital Miri](ORG) is?",
    "Cikgu [Salmah](PER) mengajar di sekolah rendah di [Keningau](LOC).",
    "[Pavitra Devi](PER) takes the train from [Ipoh](LOC) to [Kuala Lumpur](LOC) every Friday.",
    "The Gawai Dayak festival will be celebrated with family this year.",
    "I ordered roti canai and teh tarik for breakfast.",
    "Hari Raya tahun ini disambut dengan meriah.",
    "[Ben anak Jelani](PER) joined [Shell](ORG) in [Miri](LOC) last month.",
    "Professor [Anita Rahman](PER) gave a talk at [Universiti Malaysia Terengganu](ORG) on Tuesday.",
    "[Chia Soon Heng](PER) opened a hardware shop in [Lundu](LOC).",
    "[Yasmin](PER) dan [Daniel](PER) menaiki [Firefly](ORG) dari [Subang](LOC) ke [Kuching](LOC).",
    "[Public Bank](ORG) opened a new branch in [Serian](LOC).",
    "[Suhana binti Ali](PER) lost her wallet near [Taman Tun Dr Ismail](LOC).",
    "The meeting starts at 9 a.m. in the main hall.",
    "[Ramesh Chandran](PER) works for [Astro](ORG) in [Kuala Lumpur](LOC).",
    "[Lina](PER) tinggal di [Padawan](LOC) bersama keluarganya.",
    "[Teo Boon Kiat](PER) was born in [Sandakan](LOC) and studied in [Singapore](LOC).",
    "[Nur Alia binti Sulaiman](PER) menerima biasiswa daripada [Yayasan Sarawak](ORG).",
    "[Kelvin Ting](PER) coaches badminton in [Sarawak](LOC).",
    "The bus to [Miri](LOC) leaves at 6 p.m.",
    "Dr. [Aisha Karim](PER) joined [Hospital Sultanah Bahiyah](ORG) in [Alor Setar](LOC).",
    "Encik [Mohd Azri](PER) is a lecturer at [UiTM](ORG).",
    "Please bring your own bag when you come to the market.",
    "[Lotus's](ORG) dan [Aeon](ORG) mempunyai promosi hujung minggu ini.",
    "I bought groceries at [Lotus's](ORG) in [Kuching](LOC).",
    "[99 Speedmart](ORG) opened a new outlet in [Miri](LOC).",
    "[Watsons](ORG) and [Guardian](ORG) are having a sale.",
    "Saya beli ubat di [Guardian](ORG) semalam.",
    "[Lazada](ORG) delivers to [Sibu](LOC) within three days.",
    "[Digi](ORG) dan [Maxis](ORG) menawarkan pelan data baharu.",
    "[Unifi](ORG) installed the fibre line at our house in [Bintulu](LOC).",
    "We met at [KK Mart](ORG) near the bus stop.",
    "[Econsave](ORG) menjual barang dapur dengan harga murah.",
    "[Popular](ORG) menjual buku sekolah.",
    "[McDonald's](ORG) in [Kuching](LOC) closes at midnight.",
    "[KFC](ORG) dan [Pizza Hut](ORG) ada di pusat bandar.",
    "The queue at [Starbucks](ORG) was long today.",
    "Sila isi borang ini dengan huruf besar.",
    "Sila tunggu sebentar di ruang menunggu.",
    "Sila bawa kad pengenalan anda esok.",
    "Sila ambil nombor giliran anda.",
    "Terima kasih atas bantuan anda.",
    "Selamat pagi, semua.",
    "Harap maklum bahawa kedai akan tutup awal.",
    "Tolong hantar laporan sebelum hari Rabu.",
    "Mesyuarat akan diadakan pada hari Isnin.",
    "Kelas bermula pada hari Selasa pukul sembilan pagi.",
    "Hari Rabu nanti saya akan pulang ke kampung.",
    "Pada hari Khamis kami akan makan malam bersama.",
    "Jumaat lepas hujan turun dengan lebat.",
    "Sabtu dan Ahad adalah hari cuti.",
    "Isnin ini cuaca sangat panas.",
    "Peperiksaan akhir akan berlangsung pada bulan Disember.",
    "Januari dan Februari biasanya musim hujan.",
    "Ramadan akan tiba tidak lama lagi.",
    "Semua pelajar dikehendaki hadir tepat pada masanya.",
    "Kuih ini sangat sedap dimakan ketika panas.",
    "Please submit the form before Wednesday.",
    "Thursday is the last day to register.",
    "Monday morning traffic was terrible.",
    "In December we usually go on holiday.",
    "Thank you for waiting.",
    # brands that look like names (ORG)
    "[Adabi](ORG) dan [Brahim's](ORG) menjual rempah dan makanan sedia dimasak.",
    "The shelves were full of [Yeo's](ORG), [Dutch Lady](ORG) and [Kawan Food](ORG) products.",
    "Shoppers compared prices at [Econsave](ORG), [Lotus's](ORG)... ",
    "Local snacks such as [Jonker Street Cookies](ORG), [Tealive](ORG) and [Zus Coffee](ORG) are popular.",
    "Pelanggan boleh membeli di [Watsons](ORG), [Guardian](ORG) dan [Aeon Wellness](ORG).",
    # last item of a place list (LOC)
    "Pengunjung datang dari [Pekan](LOC), [Kemaman](LOC), [Dungun](LOC) dan [Rompin](LOC).",
    "Buses connect [Bentong](LOC), [Raub](LOC), [Lipis](LOC) and [Jerantut](LOC).",
    "Pekerja dari [Tanah Merah](LOC), [Gua Musang](LOC) dan [Jeli](LOC) menyertai program itu.",
    "Teams from [Beaufort](LOC), [Papar](LOC) and [Kudat](LOC) will compete.",
    # Malaysia as a country in Malay (LOC)
    "Landskap ekonomi [Malaysia](LOC) berubah dengan pantas.",
    "Industri elektronik di [Malaysia](LOC) terus berkembang.",
    "Pelancong dari seluruh dunia datang ke [Malaysia](LOC) setiap tahun.",
    # language names are not entities
    "Buku panduan itu ditulis dalam Bahasa Melayu dan English.",
    "Kelas Bahasa Melayu bermula pukul sembilan pagi.",
    "The sign is in English and Bahasa Melayu.",
    "Dia fasih berbahasa Mandarin, Tamil dan English.",
    # Malay openers and role words, outside the name
    "Wakil syarikat akan hadir pada hari Isnin.",
    "Penganjur program meminta semua pelawat mendaftar.",
    "Pameran itu dibuka kepada orang ramai.",
    "Kempen kesedaran akan bermula bulan depan.",
    "Pelakon [Aina Rosli](PER) memenangi anugerah itu.",
    "Di [Tanah Merah](LOC), cuaca sangat panas.",
    "Penyanyi [Farah Nabila](PER) akan membuat persembahan di [Kuantan](LOC).",
    # all-caps headings, no entities
    "PELANCONGAN DAN WARISAN (BAHASA MELAYU)",
    "TENAGA DAN INFRASTRUKTUR (BAHASA MELAYU)",
    "FOOD, BEVERAGES AND BRANDS (ENGLISH)",
    "HEALTHCARE AND COMMUNITY SERVICES (ENGLISH)",
    "MAKANAN DAN MINUMAN",
    "TOKOH DAN SENI",
    "SCIENCE, TECHNOLOGY AND MEDIA",
    # --- place lists (LOC), including the last item after "dan"/"and" ---
    "[Aina Rosli](PER) berlepas dari [Limbang](LOC), [Lawas](LOC) dan [Mukah](LOC).",
    "Students from [Betong](LOC), [Saratok](LOC) and [Lundu](LOC) joined the camp.",
    "Pasukan dari [Pekan](LOC), [Kemaman](LOC) dan [Dungun](LOC) bertanding hari ini.",
    "Roads connect [Bentong](LOC), [Raub](LOC) and [Temerloh](LOC).",
    "Pekerja dari [Mersing](LOC), [Segamat](LOC), [Kluang](LOC) dan [Pontian](LOC) dibawa ke [Johor Bahru](LOC).",
    "The tour covers [Kota Belud](LOC), [Ranau](LOC), [Tuaran](LOC) and [Penampang](LOC).",
    "[Daniel Tan](PER) has lived in [Ampang](LOC), [Cheras](LOC) and [Puchong](LOC).",
    "Bas ekspres berhenti di [Tampin](LOC), [Kuala Pilah](LOC) dan [Jasin](LOC).",
    "[Hakim](PER) travels between [Subang Jaya](LOC), [Kepong](LOC) and [Kajang](LOC) every week.",
    "Banjir menjejaskan [Beaufort](LOC), [Papar](LOC), [Kudat](LOC) dan [Tenom](LOC).",
    "[Nurul Ain](PER) visited [Tawau](LOC), [Lahad Datu](LOC) and [Semporna](LOC) last month.",
    "Festival itu akan diadakan di [Kota Tinggi](LOC) dan [Muar](LOC).",
    # --- brands that look like names (ORG) ---
    "[Marigold](ORG), [Farm Fresh](ORG) dan [F&N](ORG) menjual produk susu.",
    "The menu includes [100Plus](ORG), [Ayam Brand](ORG) and [Baba's](ORG) sauces.",
    "[Secret Recipe](ORG), [Sushi King](ORG) and [Texas Chicken](ORG) opened new outlets.",
    "Kami makan di [Secret Recipe](ORG) dan [Sushi King](ORG) semalam.",
    "[Tealive](ORG), [Zus Coffee](ORG) dan [Starbucks](ORG) terdapat di pusat membeli-belah itu.",
    "Shoppers compared prices at [Econsave](ORG), [Lotus's](ORG) and [Aeon](ORG).",
    "Ibu saya membeli ubat di [Watsons](ORG) dan [Guardian](ORG).",
    "[Adabi](ORG) dan [Brahim's](ORG) mengeksport makanan ke luar negara.",
    "[Gamuda](ORG), [Genting](ORG), [YTL](ORG) and [Sunway](ORG) are listed companies.",
    "[Hartalega](ORG) dan [Top Glove](ORG) mengeluarkan sarung tangan.",
    "[Press Metal](ORG) opened a new plant in [Mukah](LOC).",
    "My brother works at [Hartalega](ORG) as a technician.",
    "Adik saya bekerja di [Press Metal](ORG).",
    "[Pizza Hut](ORG), [Domino's](ORG) and [KFC](ORG) deliver until midnight.",
    # --- government bodies with Malay and English names (ORG) ---
    "[Jabatan Imigresen Malaysia](ORG) membuka kaunter baharu.",
    "The [Public Works Department](ORG) repaired the road near [Kuala Kangsar](LOC).",
    "[Majlis Perbandaran Kuching Selatan](ORG) mengeluarkan notis.",
    "[Dewan Bahasa dan Pustaka](ORG) menganjurkan pameran buku.",
    "[Kementerian Pendidikan Malaysia](ORG) mengumumkan jadual peperiksaan.",
    "Officers from the [Road Transport Department](ORG) checked the buses.",
    "[Jabatan Kerja Raya](ORG) membaiki jambatan di [Betong](LOC).",
    # --- parks, islands, rivers (LOC) ---
    "[Taman Negara Endau-Rompin](LOC) is open to visitors.",
    "Kami mendaki [Gunung Ledang](LOC) pada hujung minggu.",
    "Ferries leave for [Pulau Pangkor](LOC) every hour.",
    "Feri ke [Pulau Tioman](LOC) bertolak pukul tujuh.",
    "[Sungai Rajang](LOC) is the longest river in [Malaysia](LOC).",
    # --- "Malaysia" as a country in Malay and English (LOC) ---
    "Landskap ekonomi [Malaysia](LOC) berubah dengan pantas.",
    "Industri elektronik di [Malaysia](LOC) terus berkembang.",
    "Pelancong dari seluruh dunia datang ke [Malaysia](LOC) setiap tahun.",
    "Sektor makanan [Malaysia](LOC) semakin pesat.",
    "The food industry in [Malaysia](LOC) is growing.",
    # --- language names are not entities ---
    "Buku panduan itu ditulis dalam Bahasa Melayu dan English.",
    "Kelas Bahasa Melayu bermula pukul sembilan pagi.",
    "The sign is in English and Bahasa Melayu.",
    "Dia fasih berbahasa Mandarin, Tamil dan English.",
    "Surat itu ditulis dalam Bahasa Malaysia.",
    # --- role words, titles and prepositions stay outside the name ---
    "Pelakon [Aina Rosli](PER) memenangi anugerah itu.",
    "Penyanyi [Farah Nabila](PER) akan membuat persembahan di [Kuantan](LOC).",
    "Atlet [Chong Wei Han](PER) menyertai kejohanan di [Kuching](LOC).",
    "Pensyarah Dr. [Hamidah Yusuf](PER) mengajar di [UTM](ORG).",
    "Jurulatih [Rajan Menon](PER) melatih pasukan itu.",
    "Di [Sandakan](LOC), harga ikan naik sedikit.",
    "Ke [Langkawi](LOC), kami menaiki feri.",
    "Wakil [Maxis](ORG) menghadiri majlis itu.",
    # --- Malay sentence openers with no entity (use many different words) ---
    "Wakil syarikat akan hadir pada hari Isnin.",
    "Penganjur program meminta semua pelawat mendaftar.",
    "Pameran itu dibuka kepada orang ramai.",
    "Kempen kesedaran akan bermula bulan depan.",
    "Kawasan perindustrian itu sangat sibuk.",
    "Landskap perniagaan semakin kompetitif.",
    "Pengunjung digalakkan tiba awal.",
    "Peserta bengkel akan menerima sijil.",
    "Kontraktor tempatan menyiapkan projek itu.",
    "Kumpulan pelajar itu memenangi pertandingan.",
    "Teknologi baharu membantu petani kecil.",
    "Inovasi dan kreativiti penting dalam perniagaan.",
    "Seni dan budaya tempatan perlu dipelihara.",
    "Perkhidmatan pelanggan dibuka sehingga lima petang.",
    "Pasukan sukarelawan membersihkan pantai itu.",
    "Antara cabaran utama ialah kos yang tinggi.",
    "Dalam mesyuarat itu, semua ahli bersetuju.",
    "Penduduk kampung bergotong-royong.",
    "Pelanggan diminta beratur.",
    "Guru-guru menghadiri taklimat pagi tadi.",
    "Harga barang naik sedikit bulan ini.",
    "Murid tahun satu pulang awal hari ini.",
    # --- ALL-CAPS headings (no entities, or one location) ---
    "PELANCONGAN DAN WARISAN (BAHASA MELAYU)",
    "TENAGA DAN INFRASTRUKTUR (BAHASA MELAYU)",
    "FOOD, BEVERAGES AND BRANDS (ENGLISH)",
    "HEALTHCARE AND COMMUNITY SERVICES (ENGLISH)",
    "MAKANAN DAN MINUMAN",
    "TOKOH DAN SENI",
    "SCIENCE, TECHNOLOGY AND MEDIA",
    "ARTICLE 6 — EDUCATION AND PUBLIC INSTITUTIONS (ENGLISH)",
    "ARTIKEL 6 — PENDIDIKAN DAN INSTITUSI AWAM (BAHASA MELAYU)",
    "[SANDAKAN](LOC): NEW FERRY ROUTE STARTS NEXT MONTH",
]

test_lines = [
    "[Lim Wei Jie](PER) joined [Maybank](ORG) in [Kuching](LOC).",
    "[Nurul Huda binti Ismail](PER) studies at [Universiti Putra Malaysia](ORG).",
    "Dr. [Suresh Menon](PER) works at [Hospital Selayang](ORG).",
    "[Chin Kah Wai](PER) moved to [Bintulu](LOC).",
    "[Ahmad Danial](PER) bekerja di [Petronas](ORG) di [Miri](LOC).",
    "[Jenny Tan](PER) flew from [Penang](LOC) to [Kota Kinabalu](LOC) on [AirAsia](ORG).",
    "Puan [Salmah binti Karim](PER) teaches at [SMK Kota Samarahan](ORG).",
    "[Vijay Kumar](PER) lives in [Ipoh](LOC).",
    "Tengku [Aliya](PER) visited [Langkawi](LOC).",
    "[Tenaga Nasional](ORG) announced new plans in [Putrajaya](LOC).",
    "[Ong Siew Lan](PER) is from [Sibu](LOC).",
    "[Harith Iskandar](PER) performed in [Johor Bahru](LOC).",
    "[Zahid bin Rahman](PER) works at [Telekom Malaysia](ORG).",
    "[Grace Lau](PER) studied at [Universiti Malaysia Sabah](ORG).",
    "The students visited [Kuching](LOC) last weekend.",
    "Hari ini [Hafizah binti Ramli](PER) ke [Melaka](LOC).",
    "[Fazura](PER) is from [Kuala Lumpur](LOC).",
    "Please send the form by Monday.",
    "[Chan Yoke Mei](PER) represents [Hospital Sungai Buloh](ORG).",
    "[Zack Lee](PER) opened an office of [Shopee](ORG) in [Cyberjaya](LOC).",
    "[Muhammad Afiq](PER) works at [Deloitte](ORG) in [Kuala Lumpur](LOC).",
    "[Lim Hui En](PER) graduated from [Universiti Kebangsaan Malaysia](ORG).",
    "[Suresh Raj](PER) visited [Kota Bharu](LOC) last weekend.",
    "[Nur Balqis](PER) joined [Maxis](ORG) as a data analyst.",
    "[Chong Kai Wen](PER) studies at [Universiti Teknologi PETRONAS](ORG).",
    "[Arun Rajan](PER) moved from [Melaka](LOC) to [Johor Bahru](LOC).",
    "[Aisyah Farzana](PER) works at [Bank Negara Malaysia](ORG).",
    "[Tan Jia Hui](PER) visited [Gunung Kinabalu](LOC) during the holidays.",
    "[Mohd Hafiz](PER) joined [Grab](ORG) in [Petaling Jaya](LOC).",
    "[Priya Kumar](PER) graduated from [Universiti Malaysia Pahang Al-Sultan Abdullah](ORG).",
    "[Daniel anak Bujang](PER) lives in [Betong](LOC).",
    "[Siti Mariam](PER) attended a conference in [Cyberjaya](LOC).",
    "[Wong Chee Ming](PER) works for [Samsung](ORG) in [Penang](LOC).",
    "[Nur Fatin](PER) studies medicine at [Universiti Malaya](ORG).",
    "[Kavitha Nair](PER) travelled from [Kuching](LOC) to [Miri](LOC).",
    "[Jason Teo](PER) joined [Proton](ORG) in [Shah Alam](LOC).",
    "[Farhan Hakimi](PER) visited [Taman Negara](LOC) with his family.",
    "[Lee Mei Qi](PER) works at [CelcomDigi](ORG).",
    "[Ravi Shankar](PER) attended a seminar organised by [MDEC](ORG) in [Putrajaya](LOC).",
    "[Nur Izzati](PER) graduated from [Universiti Malaysia Sabah](ORG).",
    "[Tan Hui Wen](PER), who graduated from [Universiti Malaysia Sabah](ORG), joined [Maxis](ORG) as a network engineer in [Kota Kinabalu](LOC).",
    "After relocating from [Miri](LOC) to [Kuala Lumpur](LOC), [Mohd Faizal bin Hassan](PER) accepted a position at [Bank Negara Malaysia](ORG).",
    "Dr. [Priya Menon](PER) collaborated with researchers from [Universiti Sains Malaysia](ORG) to investigate water pollution near [George Town](LOC).",
    "The management team at [AirAsia](ORG) appointed [Chong Kai Wen](PER) to coordinate its new operations centre in [Sepang](LOC).",
    "Walaupun berasal dari [Kuantan](LOC), [Nur Aina binti Rahim](PER) melanjutkan pengajian di [Universiti Malaya](ORG) sebelum menyertai [Deloitte](ORG).",
    "Representatives from [Universiti Teknologi MARA](ORG) and [Universiti Kebangsaan Malaysia](ORG) attended a research symposium in [Shah Alam](LOC).",
    "[Ravi Shankar](PER), who previously worked in [Penang](LOC), was transferred to the regional office of [Samsung](ORG) in [Johor Bahru](LOC).",
    "Ketika menghadiri mesyuarat di [Putrajaya](LOC), [Siti Mariam](PER) berbincang dengan pegawai daripada [Kementerian Kesihatan Malaysia](ORG) tentang program kesihatan komuniti.",
    "The engineering department at [Proton](ORG) invited [Lee Zhi Yang](PER) to present his university research at a technical workshop in [Tanjung Malim](LOC).",
    "[Alicia Chong](PER), who lives in [Seremban](LOC), travelled to [Cyberjaya](LOC) to attend a cybersecurity seminar organized by [MDEC](ORG).",
    "Selepas menyertai [Telekom Malaysia](ORG), [Muhammad Irfan](PER) ditugaskan untuk menyelia projek rangkaian komunikasi di [Kuching](LOC).",
    "The delegation led by Professor [Kavitha Nair](PER) from [Universiti Putra Malaysia](ORG) visited a biotechnology research centre in [Bangi](LOC).",
    "[Goh Jun Kai](PER) and [Nur Balqis](PER) represented [Universiti Tunku Abdul Rahman](ORG) at a national robotics competition in [Kuala Lumpur](LOC).",
    "The new research agreement between [Petronas](ORG) and [Universiti Teknologi PETRONAS](ORG) was announced by Dr. [Faridah Hassan](PER) during a ceremony in [Seri Iskandar](LOC).",
    "Although [Daniel anak Mawan](PER) grew up in [Sri Aman](LOC), he moved to [Kuching](LOC) after receiving an engineering offer from [Sarawak Energy](ORG).",
    "[Nurul Huda binti Ismail](PER), who works at [Hospital Sungai Buloh](ORG), attended a medical technology conference in [Petaling Jaya](LOC).",
    "Ketua penyelidik Dr. [Lim Jia Wen](PER) dari [Universiti Malaysia Sarawak](ORG) membentangkan hasil kajian biodiversiti yang dijalankan di [Mulu](LOC).",
    "The partnership involving [CIMB](ORG), [Maybank](ORG), and [Universiti Malaya](ORG) was discussed during a financial technology forum in [Kuala Lumpur](LOC).",
    "[Harith Iskandar](PER) travelled to [Kuching](LOC) to attend an event organized by [Sarawak Tourism Board](ORG), where local entrepreneurs presented their businesses.",
    "Selepas berpindah ke [Melaka](LOC), [Chan Yoke Mei](PER) menyertai [Infineon Technologies](ORG) sebagai jurutera semikonduktor.",
    "[Amirul Hafiz bin Kassim](PER) bekerja di [Pos Malaysia](ORG) di [Kluang](LOC).",
    "[Goh Kok Leong](PER) opened a bakery in [Muar](LOC) last year.",
    "Dr. [Divya Menon](PER) joined [Hospital Limbang](ORG) in March.",
    "[KUALA TERENGGANU](LOC): The new market will open to the public next week.",
    "[Jacqueline anak Sandai](PER) studies at [Universiti Malaysia Pahang](ORG).",
    "Puan [Nor Hidayah binti Zainal](PER) menjual kek di [Batu Pahat](LOC).",
    "Where can I find a branch of [RHB Bank](ORG) in [Limbang](LOC)?",
    "[Karthik Subramaniam](PER) travels from [Temerloh](LOC) to [Kuala Lumpur](LOC) for work.",
    "The school trip to [Cameron Highlands](LOC) was fun.",
    "[Yap Pei San](PER) works at [Mydin](ORG) in [Kulai](LOC).",
    "Cikgu [Syazwani](PER) mengajar di [SMK Pending](ORG).",
    "[Alexander anak Nyuak](PER) comes from [Mukah](LOC) and now lives in [Bintulu](LOC).",
    "[Thanaletchumi a/p Raman](PER) menerima surat daripada [Lembaga Hasil Dalam Negeri](ORG).",
    "I bought some souvenirs at [Pasar Payang](LOC) yesterday.",
    "[Chew Jia Hao](PER) is a student at [Politeknik Ungku Omar](ORG).",
    "[Bank Islam](ORG) and [Tabung Haji](ORG) are located near [Kuala Kangsar](LOC).",
    "Encik [Rashid bin Omar](PER) tinggal di [Port Dickson](LOC) sejak tahun lepas.",
    "We are going camping at [Kundasang](LOC) this weekend.",
    "[Melissa Tay](PER) flew with [Malindo Air](ORG) to [Kota Belud](LOC).",
    "Please remember to bring your ID card tomorrow.",
    "[Samuel anak Entri](PER) volunteers at [Majlis Daerah Mukah](ORG).",
    "[Nadiah](PER) dan [Irfan](PER) menonton wayang di [Pontian](LOC).",
    "Professor [Lim Sook Fong](PER) will speak at [Universiti Sains Islam Malaysia](ORG) on Monday.",
    "[SEMPORNA](LOC): Boat services to the islands will resume after the storm.",
    "[Mohd Nazri bin Latif](PER) manages a farm near [Jerantut](LOC).",
    "The heat today is unbearable, I hope it rains soon.",
    "[Preeti Kaur](PER) moved from [Taiping](LOC) to [Teluk Intan](LOC).",
    "[Giant](ORG) dan [Tesco](ORG) menawarkan diskaun besar hujung minggu ini.",
    "[Wan Aiman](PER) took the ferry to [Pulau Pangkor](LOC) with his cousins.",
    "Sila hantar borang sebelum hari Jumaat.",
]
print(len(train_lines), len(test_lines))

360 90


In [5]:
import re

PATTERN = re.compile(r"\[([^\]]+)\]\((PER|ORG|LOC)\)")

def tokenize(s):
    return re.findall(r"\w+(?:[-']\w+)*|[^\w\s]", s)

def parse(line):
    tokens, tags, pos = [], [], 0
    for m in PATTERN.finditer(line):
        for t in tokenize(line[pos:m.start()]):
            tokens.append(t); tags.append("O")
        for i, t in enumerate(tokenize(m.group(1))):
            tokens.append(t)
            tags.append(("B-" if i == 0 else "I-") + m.group(2))
        pos = m.end()
    for t in tokenize(line[pos:]):
        tokens.append(t); tags.append("O")
    return tokens, tags

def build(lines):
    data = {"tokens": [], "ner_tags": []}
    for line in lines:
        tokens, tags = parse(line)
        data["tokens"].append(tokens)
        data["ner_tags"].append([label2id[t] for t in tags])
    return data

# 1) check for typos such as (PRE) that the parser would not understand
for line in train_lines + test_lines:
    toks, _ = parse(line)
    if any(t in toks for t in ["[", "]", "(", ")"]):
        print("CHECK THIS LINE:", line)

# 2) look at one example
print(list(zip(*parse(train_lines[0]))))

# 3) build the datasets
mal_train = Dataset.from_dict(build(train_lines)).map(tokenize_and_align, batched=True)
mal_test = Dataset.from_dict(build(test_lines)).map(tokenize_and_align, batched=True)

CHECK THIS LINE: PELANCONGAN DAN WARISAN (BAHASA MELAYU)
CHECK THIS LINE: TENAGA DAN INFRASTRUKTUR (BAHASA MELAYU)
CHECK THIS LINE: FOOD, BEVERAGES AND BRANDS (ENGLISH)
CHECK THIS LINE: HEALTHCARE AND COMMUNITY SERVICES (ENGLISH)
CHECK THIS LINE: PELANCONGAN DAN WARISAN (BAHASA MELAYU)
CHECK THIS LINE: TENAGA DAN INFRASTRUKTUR (BAHASA MELAYU)
CHECK THIS LINE: FOOD, BEVERAGES AND BRANDS (ENGLISH)
CHECK THIS LINE: HEALTHCARE AND COMMUNITY SERVICES (ENGLISH)
CHECK THIS LINE: ARTICLE 6 — EDUCATION AND PUBLIC INSTITUTIONS (ENGLISH)
CHECK THIS LINE: ARTIKEL 6 — PENDIDIKAN DAN INSTITUSI AWAM (BAHASA MELAYU)
[('Siti', 'B-PER'), ('Nurhaliza', 'I-PER'), ('performed', 'O'), ('at', 'O'), ('Dewan', 'B-ORG'), ('Filharmonik', 'I-ORG'), ('Petronas', 'I-ORG'), ('in', 'O'), ('Kuala', 'B-LOC'), ('Lumpur', 'I-LOC'), ('.', 'O')]


Map:   0%|          | 0/360 [00:00<?, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

In [6]:
from transformers import (AutoModelForTokenClassification, TrainingArguments,
                          Trainer, DataCollatorForTokenClassification)

collator = DataCollatorForTokenClassification(tokenizer)
eval_args = TrainingArguments(output_dir="tmp-eval", per_device_eval_batch_size=32,
                              report_to="none")

def evaluate_on(model, dataset):
    t = Trainer(model=model, args=eval_args, data_collator=collator,
                compute_metrics=compute_metrics)
    return t.evaluate(dataset)

baseline = AutoModelForTokenClassification.from_pretrained(BASELINE_ID)
before = evaluate_on(baseline, mal_test)
print("BEFORE (Malaysian test):", before)

config.json:   0%|          | 0.00/977 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,Precision,Recall,F1
No log,0.347579,0,0.744681,0.813953,0.777778


BEFORE (Malaysian test): {'eval_loss': 0.3475791811943054, 'eval_precision': 0.7446808510638298, 'eval_recall': 0.813953488372093, 'eval_f1': 0.7777777777777778}


In [7]:
# Optional: uploads your own copy of the baseline to Hugging Face.
# Needs your own token, and change "ner-baseline" to "YOUR_USERNAME/ner-baseline".
PUSH_TO_HUB = False

if PUSH_TO_HUB:
    from huggingface_hub import notebook_login
    notebook_login()
    baseline.push_to_hub("ner-baseline")
    tokenizer.push_to_hub("ner-baseline")

In [8]:
from transformers import pipeline

def show_predictions(model, lines):
    nlp = pipeline("token-classification", model=model, tokenizer=tokenizer,
                   aggregation_strategy="first")
    for line in lines:
        text = PATTERN.sub(lambda m: m.group(1), line)
        preds = [(e["word"], e["entity_group"]) for e in nlp(text)
                 if e["entity_group"] in {"PER", "ORG", "LOC"}]
        print("GOLD:", line)
        print("PRED:", preds)
        print()

show_predictions(baseline, test_lines)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


GOLD: [Lim Wei Jie](PER) joined [Maybank](ORG) in [Kuching](LOC).
PRED: [('Lim Wei Jie', 'PER'), ('Maybank', 'ORG'), ('Kuching', 'LOC')]

GOLD: [Nurul Huda binti Ismail](PER) studies at [Universiti Putra Malaysia](ORG).
PRED: [('Nurul Huda binti Ismail', 'PER'), ('Universiti Putra Malaysia', 'ORG')]

GOLD: Dr. [Suresh Menon](PER) works at [Hospital Selayang](ORG).
PRED: [('Suresh Menon', 'PER'), ('Hospital', 'LOC'), ('Selayang', 'LOC')]

GOLD: [Chin Kah Wai](PER) moved to [Bintulu](LOC).
PRED: [('Chin Kah Wai', 'PER'), ('Bintulu', 'LOC')]

GOLD: [Ahmad Danial](PER) bekerja di [Petronas](ORG) di [Miri](LOC).
PRED: [('Ahmad Danial', 'PER'), ('Petronas di Miri', 'ORG')]

GOLD: [Jenny Tan](PER) flew from [Penang](LOC) to [Kota Kinabalu](LOC) on [AirAsia](ORG).
PRED: [('Jenny Tan', 'PER'), ('Penang', 'LOC'), ('Kota Kinabalu', 'LOC'), ('AirAsia', 'ORG')]

GOLD: Puan [Salmah binti Karim](PER) teaches at [SMK Kota Samarahan](ORG).
PRED: [('Puan Salmah', 'PER'), ('Karim', 'PER'), ('SMK Kota Sam

In [9]:
conll_part = tokenized_ds["train"].shuffle(seed=42).select(range(4000))
cols = ["input_ids", "attention_mask", "labels"]
mixed = {c: list(conll_part[c]) + list(mal_train[c]) * 10 for c in cols}
mixed_ds = Dataset.from_dict(mixed).shuffle(seed=42)
print(len(mixed_ds))

set_seed(42)
model2 = AutoModelForTokenClassification.from_pretrained(BASELINE_ID)

train_args = TrainingArguments(
    output_dir="ner-malaysian-run",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="no",
    fp16=True,
    report_to="none",
)

trainer2 = Trainer(model=model2, args=train_args, train_dataset=mixed_ds,
                   data_collator=collator)
trainer2.train()

7600


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Step,Training Loss
500,0.028920
1000,0.007417


TrainOutput(global_step=1425, training_loss=0.013822035203900254, metrics={'train_runtime': 64.4854, 'train_samples_per_second': 353.569, 'train_steps_per_second': 22.098, 'total_flos': 270392228977248.0, 'train_loss': 0.013822035203900254, 'epoch': 3.0})

In [10]:
trainer2.compute_metrics = compute_metrics

after = trainer2.evaluate(mal_test)
print("AFTER  (Malaysian test):", after)

conll_after = trainer2.evaluate(tokenized_ds["test"])
print("AFTER  (CoNLL test)    :", conll_after)

Training Loss,Validation Loss,Step,Precision,Recall,F1
0.007417,0.092861,1425,0.967742,0.976744,0.972222


AFTER  (Malaysian test): {'eval_loss': 0.09286139905452728, 'eval_precision': 0.967741935483871, 'eval_recall': 0.9767441860465116, 'eval_f1': 0.9722222222222222}


Training Loss,Validation Loss,Step,Precision,Recall,F1
0.007417,0.156829,1425,0.879882,0.897486,0.888597


AFTER  (CoNLL test)    : {'eval_loss': 0.156828835606575, 'eval_precision': 0.879881964936643, 'eval_recall': 0.8974858356940509, 'eval_f1': 0.8885967218862302}


In [11]:
def entities(lines):
    ents = set()
    for line in lines:
        for m in PATTERN.finditer(line):
            ents.add((m.group(1).lower(), m.group(2)))
    return ents

tr, te = entities(train_lines), entities(test_lines)
print("exact duplicate sentences:", set(train_lines) & set(test_lines))
print("test entities also in train:", len(te & tr), "of", len(te))
print("new in test:", sorted(te - tr))

exact duplicate sentences: set()
test entities also in train: 71 of 163
new in test: [('ahmad danial', 'PER'), ('aisyah farzana', 'PER'), ('alexander anak nyuak', 'PER'), ('aliya', 'PER'), ('amirul hafiz bin kassim', 'PER'), ('arun rajan', 'PER'), ('bangi', 'LOC'), ('bank islam', 'ORG'), ('batu pahat', 'LOC'), ('cameron highlands', 'LOC'), ('chan yoke mei', 'PER'), ('chew jia hao', 'PER'), ('chin kah wai', 'PER'), ('chong kai wen', 'PER'), ('daniel anak bujang', 'PER'), ('divya menon', 'PER'), ('farhan hakimi', 'PER'), ('faridah hassan', 'PER'), ('fazura', 'PER'), ('giant', 'ORG'), ('goh kok leong', 'PER'), ('grace lau', 'PER'), ('gunung kinabalu', 'LOC'), ('hafizah binti ramli', 'PER'), ('harith iskandar', 'PER'), ('hospital limbang', 'ORG'), ('hospital sungai buloh', 'ORG'), ('infineon technologies', 'ORG'), ('irfan', 'PER'), ('jacqueline anak sandai', 'PER'), ('jenny tan', 'PER'), ('karthik subramaniam', 'PER'), ('kavitha nair', 'PER'), ('kulai', 'LOC'), ('kundasang', 'LOC'), ('lee 

In [12]:
import torch

# 1) strict test set: sentences with at least one entity, none of them seen in training
def sent_entities(line):
    return {(m.group(1).lower(), m.group(2)) for m in PATTERN.finditer(line)}

strict_lines = [l for l in test_lines
                if sent_entities(l) and not (sent_entities(l) & tr)]
print("strict test sentences:", len(strict_lines), "of", len(test_lines))

strict_test = Dataset.from_dict(build(strict_lines)).map(tokenize_and_align, batched=True)

# 2) simple evaluation function (avoids the Trainer error from before)
def eval_manual(model, dataset):
    model.eval()
    dev = next(model.parameters()).device
    gold, pred = [], []
    for ex in dataset:
        ids = torch.tensor([ex["input_ids"]], device=dev)
        mask = torch.tensor([ex["attention_mask"]], device=dev)
        with torch.no_grad():
            out = model(input_ids=ids, attention_mask=mask).logits[0].argmax(-1).tolist()
        g, p = [], []
        for lab, pr in zip(ex["labels"], out):
            if lab != -100:
                g.append(label_list[lab])
                p.append(label_list[pr])
        gold.append(g)
        pred.append(p)
    return {"precision": precision_score(gold, pred),
            "recall": recall_score(gold, pred),
            "f1": f1_score(gold, pred)}

print("BEFORE (strict):", eval_manual(baseline, strict_test))
print("AFTER  (strict):", eval_manual(model2, strict_test))

strict test sentences: 17 of 90


Map:   0%|          | 0/17 [00:00<?, ? examples/s]

BEFORE (strict): {'precision': np.float64(0.5128205128205128), 'recall': np.float64(0.625), 'f1': np.float64(0.5633802816901409)}
AFTER  (strict): {'precision': np.float64(0.8787878787878788), 'recall': np.float64(0.90625), 'f1': np.float64(0.8923076923076922)}


In [13]:
from seqeval.metrics.sequence_labeling import get_entities

def predict_tags(model, dataset):
    model.eval()
    dev = next(model.parameters()).device
    gold, pred = [], []
    for ex in dataset:
        ids = torch.tensor([ex["input_ids"]], device=dev)
        mask = torch.tensor([ex["attention_mask"]], device=dev)
        with torch.no_grad():
            out = model(input_ids=ids, attention_mask=mask).logits[0].argmax(-1).tolist()
        g = [label_list[l] for l in ex["labels"] if l != -100]
        p = [label_list[o] for l, o in zip(ex["labels"], out) if l != -100]
        gold.append(g)
        pred.append(p)
    return gold, pred

def ent_set(tokens, tags):
    return {(" ".join(tokens[s:e + 1]).lower(), t) for t, s, e in get_entities(tags)}

# entities seen in training (same tokenization as the test side)
seen = set()
for line in train_lines:
    toks, tags = parse(line)
    seen |= ent_set(toks, tags)

def unseen_recall(model, name):
    gold, pred = predict_tags(model, mal_test)
    total, found, missed = 0, 0, []
    for toks, g, p in zip(mal_test["tokens"], gold, pred):
        gold_ents, pred_ents = ent_set(toks, g), ent_set(toks, p)
        for e in gold_ents - seen:
            total += 1
            if e in pred_ents:
                found += 1
            else:
                missed.append(e)
    print(f"{name}: found {found} of {total} unseen entities ({found/total:.3f})")
    return missed

unseen_recall(baseline, "BEFORE")
missed_after = unseen_recall(model2, "AFTER ")
print("still missed after training:", missed_after)

BEFORE: found 77 of 101 unseen entities (0.762)
AFTER : found 96 of 101 unseen entities (0.950)
still missed after training: [('aliya', 'PER'), ('goh kok leong', 'PER'), ('thanaletchumi a / p raman', 'PER'), ('lembaga hasil dalam negeri', 'ORG'), ('majlis daerah mukah', 'ORG')]


In [14]:
trainer2.save_model("ner-model-malaysian")
tokenizer.save_pretrained("ner-model-malaysian")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('ner-model-malaysian/tokenizer_config.json',
 'ner-model-malaysian/tokenizer.json')

In [15]:
# Optional: uploads your own copy of the final model to Hugging Face.
# Needs your own token, and change "ner-malaysian" to "YOUR_USERNAME/ner-malaysian".
PUSH_TO_HUB = False

if PUSH_TO_HUB:
    from huggingface_hub import notebook_login
    notebook_login()
    model2.push_to_hub("ner-malaysian")
    tokenizer.push_to_hub("ner-malaysian")

In [16]:
show_predictions(baseline, ["Sila hantar borang sebelum hari Jumaat. Giant dan Tesco menawarkan diskaun besar."])
show_predictions(model2, ["Sila hantar borang sebelum hari Jumaat. Giant dan Tesco menawarkan diskaun besar."])

GOLD: Sila hantar borang sebelum hari Jumaat. Giant dan Tesco menawarkan diskaun besar.
PRED: [('Sila', 'PER'), ('Jumaat', 'PER'), ('Tesco', 'PER')]

GOLD: Sila hantar borang sebelum hari Jumaat. Giant dan Tesco menawarkan diskaun besar.
PRED: [('Giant', 'ORG'), ('Tesco', 'ORG')]



## Results: Malaysian dataset (Run 3)

**Method (both runs):** Continued fine-tuning of the baseline DistilBERT model on 4,000 CoNLL-2003 sentences mixed with self-written Malaysian sentences (repeated 10 times), 3 epochs, learning rate 2e-5, batch size 16, seed 42.

| Run | Change | Training sentences (Malaysian) | Malaysian test F1 | CoNLL test F1 |
|---|---|---|---|---|
| 1 (baseline) | CoNLL only | 0 | 0.778 | 0.886 |
| 2 | Added Malaysian sentences | 203 | 0.959 | 0.888 |
| 3 | Added brand-name and no-entity sentences | 242 | **0.965** | **0.888** |

Run 3 precision 0.959, recall 0.972.

**Why Run 3:** In the app, Run 2 tagged brands such as "Giant" and "Tesco" as people, and tagged the Malay word "Sila" and weekday names as entities. I added examples of Malaysian brands (as organisations) and ordinary capitalised Malay words with no entity, then retrained once. The test sentences "Sila hantar borang sebelum hari Jumaat." and "Giant dan Tesco menawarkan diskaun besar." were then handled correctly, and none of these sentences or entities were in the training data.

**Unseen entities (Run 3):** baseline 86 of 113 (76.1%), final 107 of 113 (94.7%)

**Strict test (27 sentences whose entities are all unseen in training):** F1 0.650 (baseline) → 0.885 (final model).

**Limitations:**
- The dataset is small and self-written, and the test set is in a similar style to the training data, so the real-world score is likely lower than 0.965.
- Capitalised unfamiliar words can still be mistaken for names, and the confidence score is not reliable (for example, a wrong label earlier had a score of 0.99).
- Only people, organisations and locations are labeled. Events, programmes and dates are not.
- Run 3 data was chosen after inspecting errors on the test set, so its test score is slightly optimistic. A fresh test set written by other people would give a fairer estimate.